# Blue Catalyst POC — ESM2 Proteome Embeddings + Advanced Analytics
## Wetland MUCC v2.0.0 × Rumen PRJEB31266

This notebook has two operating modes:

- **OFFLINE MODE** (default locally): loads pre-computed embeddings from the
  validated Apolo-3 run artefacts, then applies a rich suite of advanced
  analytics and writes new figures / reports alongside existing artefacts.
- **ONLINE MODE**: runs the full pipeline — data acquisition → ESM2 embedding →
  analysis — on a machine with GPU access (Apolo-3 SLURM).

### Analytics added in this version
1. PCA variance landscape (scree + PC1×PC2 scatter)
2. KDE density contour landscapes for UMAP and t-SNE (static PNG + interactive HTML)
3. Pairwise cosine-distance heatmap (genome × genome)
4. PERMANOVA: permutation test for ecosystem separation
5. Per-genome silhouette profiles
6. Ecosystem trajectory — projection onto rumen→wetland axis
7. Enhanced bridge-genome analysis (mixing coefficient + KNN tables)
8. Comprehensive 2×2 proposal panel figure


In [ ]:
from pathlib import Path
import os
from datetime import datetime


def infer_project_root() -> Path:
    env_root = os.getenv("METHANET_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "methanet").exists():
            return p
    return cwd


PROJECT_ROOT = infer_project_root()

run_id = os.getenv("BC_RUN_ID") or datetime.now().strftime("run_%Y%m%d_%H%M%S")
_env_art = os.getenv("BC_ARTIFACTS_DIR")
ARTIFACTS_DIR = (
    Path(_env_art).expanduser().resolve()
    if _env_art
    else PROJECT_ROOT / "results" / "blue_catalyst_poc" / "runs" / run_id / "artifacts"
)

CFG = {
    "run_id": run_id,
    "out_root": ARTIFACTS_DIR,
    "data_root": PROJECT_ROOT / "data" / "blue_catalyst_poc",
    # MUCC
    "mucc_zenodo_record": os.getenv("MUCC_ZENODO_RECORD", "14532347"),
    "mucc_target_key": os.getenv("MUCC_TARGET_KEY", "MUCC_v2.0.0_HQMQ_genes.faa.zip"),
    "mucc_manual_proteome_url": os.getenv("MUCC_MANUAL_PROTEOME_URL") or None,
    # Rumen ENA
    "rumen_study": os.getenv("RUMEN_STUDY", "PRJEB31266"),
    # Sampling controls (default = maximize samples)
    "subset_mode": os.getenv("BC_SUBSET_MODE", "0") == "1",
    "subset_mucc_genomes": int(os.getenv("BC_SUBSET_MUCC", "200")),
    "subset_rumen_genomes": int(os.getenv("BC_SUBSET_RUMEN", "200")),
    "rumen_max_files_per_analysis": int(os.getenv("BC_RUMEN_MAX_PER_ANALYSIS", "3")),
    # Embedding
    "esm2_model": os.getenv("BC_ESM2_MODEL", "facebook/esm2_t33_650M_UR50D"),
    "esm2_batch_size": int(os.getenv("BC_ESM2_BATCH", "4")),
    "esm2_max_length": int(os.getenv("BC_ESM2_MAXLEN", "1022")),
    "device": os.getenv("BC_DEVICE", "auto"),
    # Data handling
    "max_proteins_per_genome": int(os.getenv("BC_MAX_PROTEINS", "2000")),
    "min_aa_len": int(os.getenv("BC_MIN_AA_LEN", "30")),
    "allow_gene_calling_fallback": os.getenv("RUMEN_ALLOW_GENE_CALLING", "1") == "1",
    # Analysis
    "umap_n_neighbors": int(os.getenv("BC_UMAP_NEIGHBORS", "20")),
    "umap_min_dist": float(os.getenv("BC_UMAP_MINDIST", "0.15")),
    "tsne_perplexity": int(os.getenv("BC_TSNE_PERPLEXITY", "20")),
    "knn_k_bridge": int(os.getenv("BC_BRIDGE_K", "15")),
}

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OFFLINE_MODE = os.getenv("BC_OFFLINE_MODE", "0") == "1"

print(f"Project root : {PROJECT_ROOT}")
print(f"Run ID       : {CFG['run_id']}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Offline mode : {OFFLINE_MODE}")
print(f"Subset mode  : {CFG['subset_mode']}")
print(f"Data root    : {CFG['data_root']}")

In [ ]:
import csv
import gzip
import io
import json
import math
import random
import re
import shutil
import subprocess
import urllib.parse
import urllib.request
import warnings
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # non-interactive backend for batch execution
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import gaussian_kde
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder

try:
    import umap as umap_pkg
    HAS_UMAP = True
except ModuleNotFoundError:
    HAS_UMAP = False
    umap_pkg = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

ECOSYSTEM_COLORS = {"wetland": "#2E86AB", "rumen": "#D62828"}

print("Imports OK")
print(f"UMAP available: {HAS_UMAP}")

In [ ]:
def http_get_json(url: str, timeout: int = 60) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return json.load(resp)


def stream_download(url: str, out_path: Path, chunk_size: int = 1 << 20) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url, timeout=120) as src, out_path.open("wb") as dst:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            dst.write(chunk)


def is_probably_protein(seq: str) -> bool:
    if not seq:
        return False
    dna_chars = set("ACGTNacgtn")
    aa_chars = set("ACDEFGHIKLMNPQRSTVWYBXZJUO*acdefghiklmnpqrstvwybxzjuo")
    dna_fraction = sum(ch in dna_chars for ch in seq) / max(1, len(seq))
    aa_fraction = sum(ch in aa_chars for ch in seq) / max(1, len(seq))
    return aa_fraction > 0.95 and dna_fraction < 0.9


def sanitize_sample_id(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def read_fasta_preview(path: Path, n_records: int = 20):
    from Bio import SeqIO
    cnt = 0
    aa_like = 0
    lengths = []
    open_fn = gzip.open if path.suffix == ".gz" else open
    with open_fn(path, "rt") as handle:
        for rec in SeqIO.parse(handle, "fasta"):
            s = str(rec.seq)
            lengths.append(len(s))
            if is_probably_protein(s):
                aa_like += 1
            cnt += 1
            if cnt >= n_records:
                break
    return {
        "checked": cnt,
        "aa_like": aa_like,
        "mean_len": float(np.mean(lengths)) if lengths else 0.0,
    }


def maybe_decompress_to_temp(src_path: Path, tmp_dir: Path) -> Path:
    if src_path.suffix != ".gz":
        return src_path
    tmp_dir.mkdir(parents=True, exist_ok=True)
    out_path = tmp_dir / src_path.with_suffix("").name
    if out_path.exists():
        return out_path
    try:
        with gzip.open(src_path, "rb") as src, out_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    except (EOFError, OSError) as e:
        out_path.unlink(missing_ok=True)
        raise RuntimeError(f"Corrupted gzip: {src_path}") from e
    return out_path


def run_prodigal_safe(nuc_fp: Path, out_fp: Path, context: str) -> bool:
    cmd = ["prodigal", "-i", str(nuc_fp), "-a", str(out_fp), "-p", "meta", "-q"]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except subprocess.CalledProcessError as e:
        if out_fp.exists() and out_fp.stat().st_size == 0:
            out_fp.unlink(missing_ok=True)
        print(f"SKIP {context}: prodigal exit {e.returncode} on {nuc_fp.name}")
        return False


def ensure_prodigal_available():
    if shutil.which("prodigal") is None:
        raise RuntimeError("prodigal not on PATH; required for gene-calling fallback.")


def select_zip_members(zip_path: Path, subset_n):
    import zipfile

    with zipfile.ZipFile(zip_path, "r") as zf:
        names = [n for n in zf.namelist() if not n.endswith("/")]
    protein_exts = (".faa", ".faa.gz", ".fa", ".fa.gz")
    nucleotide_exts = (".fna", ".fna.gz", ".fasta", ".fasta.gz")
    prot = [n for n in names if n.lower().endswith(protein_exts)]
    nuc = [n for n in names if n.lower().endswith(nucleotide_exts)]
    chosen = sorted(prot if prot else nuc)
    is_protein = bool(prot)
    if subset_n is not None:
        chosen = chosen[:subset_n]
    return chosen, is_protein


def select_diverse_rumen_subset(manifest_df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if manifest_df.empty:
        return manifest_df.copy()

    ranked = manifest_df.sort_values(["analysis_accession", "filename"]).reset_index(drop=True)
    per_analysis_cap = max(1, int(cfg["rumen_max_files_per_analysis"]))

    if cfg["subset_mode"]:
        target_n = max(1, int(cfg["subset_rumen_genomes"]))
    else:
        target_n = len(ranked)

    buckets = [
        g.head(per_analysis_cap).copy()
        for _, g in ranked.groupby("analysis_accession", dropna=False)
    ]

    if not buckets:
        return ranked.head(target_n).copy()

    chosen_parts = []
    depth = 0
    while len(chosen_parts) < target_n:
        progressed = False
        for b in buckets:
            if depth < len(b):
                chosen_parts.append(b.iloc[[depth]])
                progressed = True
                if len(chosen_parts) >= target_n:
                    break
        if not progressed:
            break
        depth += 1

    chosen = pd.concat(chosen_parts, ignore_index=True) if chosen_parts else ranked.head(target_n).copy()
    return chosen.drop_duplicates(subset=["download_url"]).head(target_n).copy()


ARTIFACT_RECORDS: list[dict] = []


def record_artifact(path: Path, category: str, description: str) -> None:
    p = Path(path)
    ARTIFACT_RECORDS.append(
        {
            "file": p.name,
            "relative_path": str(p.relative_to(ARTIFACTS_DIR)),
            "category": category,
            "description": description,
        }
    )


print("Utility functions defined")

In [ ]:
if OFFLINE_MODE:
    print("OFFLINE MODE — skipping data acquisition.")
    mucc_files = []
    mucc_download_path = None
    rumen_manifest = pd.DataFrame()
    sample_df = pd.DataFrame()
else:
    from Bio import SeqIO
    import zipfile

    mucc_dir = CFG["data_root"] / "mucc"
    mucc_dir.mkdir(parents=True, exist_ok=True)

    zenodo_url = f"https://zenodo.org/api/records/{CFG['mucc_zenodo_record']}"
    mucc_record = http_get_json(zenodo_url)
    mucc_files = mucc_record.get("files", [])
    print("MUCC record reachable:", bool(mucc_files), "| n_files:", len(mucc_files))

    # ENA PRJEB31266
    base = "https://www.ebi.ac.uk/ena/portal/api/search"
    query = f'study_accession="{CFG["rumen_study"]}" AND analysis_type="SEQUENCE_ASSEMBLY"'
    params = {
        "result": "analysis",
        "query": query,
        "fields": "analysis_accession,scientific_name,submitted_ftp",
        "format": "tsv",
        "limit": "5",
    }
    ena_url = base + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(ena_url, timeout=60) as resp:
        ena_tsv = resp.read().decode("utf-8", errors="ignore")
    rows = [r for r in ena_tsv.splitlines() if r.strip()]
    print("ENA reachable:", len(rows) > 1, "| rows:", max(0, len(rows) - 1))

    # MUCC download
    mucc_file_map = {f["key"]: f for f in mucc_files}
    mucc_target = CFG["mucc_target_key"]
    mucc_download_path = None
    for key in [mucc_target, "Methanoregula_MAGs_DB.zip"]:
        if key in mucc_file_map:
            entry = mucc_file_map[key]
            mucc_download_path = mucc_dir / key
            if not mucc_download_path.exists():
                print("Downloading MUCC:", key)
                stream_download(entry["links"]["self"], mucc_download_path)
            else:
                print("MUCC already exists:", mucc_download_path)
            break
    if mucc_download_path is None and CFG["mucc_manual_proteome_url"]:
        mucc_download_path = mucc_dir / Path(
            urllib.parse.urlparse(CFG["mucc_manual_proteome_url"]).path
        ).name
        if not mucc_download_path.exists():
            stream_download(CFG["mucc_manual_proteome_url"], mucc_download_path)

    if mucc_download_path and mucc_download_path.exists():
        record_artifact(mucc_download_path, "input_cache", "Local MUCC archive downloaded or reused")

    # Rumen manifest
    rumen_dir = CFG["data_root"] / "rumen"
    rumen_dir.mkdir(parents=True, exist_ok=True)
    params2 = dict(params)
    params2.update({
        "fields": "analysis_accession,scientific_name,analysis_alias,submitted_ftp",
        "limit": "0",
    })
    del params2["limit"]
    params2["limit"] = "0"
    manifest_url = base + "?" + urllib.parse.urlencode(params2)
    with urllib.request.urlopen(manifest_url, timeout=180) as resp:
        tsv = resp.read().decode("utf-8", errors="ignore")
    rumen_rows = []
    reader = csv.DictReader(io.StringIO(tsv), delimiter="\t")
    for row in reader:
        ftp_field = (row.get("submitted_ftp") or "").strip()
        for rel in ftp_field.split(";"):
            rel = rel.strip()
            if not rel:
                continue
            rumen_rows.append({
                "analysis_accession": row.get("analysis_accession", ""),
                "scientific_name": row.get("scientific_name", ""),
                "analysis_alias": row.get("analysis_alias", ""),
                "submitted_ftp_rel": rel,
                "download_url": "https://" + rel,
                "filename": Path(rel).name,
            })

    rumen_manifest = pd.DataFrame(rumen_rows).drop_duplicates(subset=["download_url"])
    rumen_manifest["ecosystem"] = "rumen"
    rumen_manifest["domain"] = np.where(
        rumen_manifest["scientific_name"].str.contains("archae|euryarchae", case=False, na=False),
        "Archaea",
        "Bacteria",
    )

    rumen_manifest_path = ARTIFACTS_DIR / "prjeb31266_analysis_manifest.tsv"
    rumen_manifest.to_csv(rumen_manifest_path, sep="\t", index=False)
    record_artifact(rumen_manifest_path, "metadata", "Full ENA analysis manifest for PRJEB31266")

    rumen_subset = select_diverse_rumen_subset(rumen_manifest, CFG)
    rumen_subset_path = ARTIFACTS_DIR / "prjeb31266_selected_subset.tsv"
    rumen_subset.to_csv(rumen_subset_path, sep="\t", index=False)
    record_artifact(
        rumen_subset_path,
        "metadata",
        "Diversity-aware selected rumen subset (stratified by analysis accession)",
    )

    rumen_dir_raw = rumen_dir / "raw"
    rumen_dir_raw.mkdir(parents=True, exist_ok=True)

    for _, r in rumen_subset.iterrows():
        out_fp = rumen_dir_raw / r["filename"]
        if not out_fp.exists():
            stream_download(r["download_url"], out_fp)

    print("Rumen manifest rows:", len(rumen_manifest))
    print("Rumen selected rows:", len(rumen_subset))
    print("Unique analyses in selected set:", rumen_subset["analysis_accession"].nunique())

In [ ]:
if OFFLINE_MODE:
    print("OFFLINE MODE — skipping proteome preparation.")
else:
    from Bio import SeqIO
    import zipfile

    proteome_dir = CFG["data_root"] / "proteomes"
    proteome_dir.mkdir(parents=True, exist_ok=True)
    sample_records = []

    # MUCC
    if mucc_download_path and mucc_download_path.exists():
        mucc_extract_dir = CFG["data_root"] / "mucc" / "extracted"
        mucc_extract_dir.mkdir(exist_ok=True, parents=True)
        subset_n = CFG["subset_mucc_genomes"] if CFG["subset_mode"] else None
        if str(mucc_download_path).endswith(".zip"):
            chosen_members, zip_is_protein = select_zip_members(mucc_download_path, subset_n)
            print(f"MUCC zip: {len(chosen_members)} members | protein={zip_is_protein}")
            with zipfile.ZipFile(mucc_download_path, "r") as zf:
                for member in chosen_members:
                    target_path = mucc_extract_dir / member
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    if not target_path.exists():
                        with zf.open(member) as src, target_path.open("wb") as dst:
                            shutil.copyfileobj(src, dst)
                    sample = sanitize_sample_id(
                        Path(member).stem.replace(".faa", "").replace(".fa", "")
                    )
                    out_fp = proteome_dir / f"mucc__{sample}.faa"
                    if zip_is_protein:
                        try:
                            if target_path.suffix == ".gz":
                                with gzip.open(target_path, "rb") as s, out_fp.open("wb") as d:
                                    shutil.copyfileobj(s, d)
                            else:
                                shutil.copyfile(target_path, out_fp)
                        except (EOFError, OSError) as e:
                            print(f"SKIP MUCC {target_path}: {e}")
                            continue
                        sample_records.append(
                            dict(
                                sample=out_fp.stem,
                                source="mucc",
                                ecosystem="wetland",
                                domain="Unknown",
                                source_analysis_accession="",
                                proteome_faa=str(out_fp),
                            )
                        )
                    elif CFG["allow_gene_calling_fallback"]:
                        ensure_prodigal_available()
                        try:
                            nuc = maybe_decompress_to_temp(target_path, proteome_dir / "_tmp_mucc")
                        except RuntimeError as e:
                            print(f"SKIP MUCC nuc {target_path}: {e}")
                            continue
                        if not run_prodigal_safe(nuc, out_fp, "MUCC"):
                            continue
                        sample_records.append(
                            dict(
                                sample=out_fp.stem,
                                source="mucc",
                                ecosystem="wetland",
                                domain="Unknown",
                                source_analysis_accession="",
                                proteome_faa=str(out_fp),
                            )
                        )

    # Rumen
    rumen_raw_dir = CFG["data_root"] / "rumen" / "raw"
    if rumen_raw_dir.exists():
        selected_df = pd.DataFrame()
        subset_path = ARTIFACTS_DIR / "prjeb31266_selected_subset.tsv"
        if subset_path.exists():
            selected_df = pd.read_csv(subset_path, sep="\t")

        if not selected_df.empty:
            rumen_files = [
                rumen_raw_dir / fn
                for fn in selected_df["filename"].astype(str).tolist()
                if (rumen_raw_dir / fn).exists()
            ]
            source_meta = selected_df[["filename", "domain", "analysis_accession"]].drop_duplicates()
        else:
            rumen_files = sorted(rumen_raw_dir.glob("*.fa.gz"))
            source_meta = pd.DataFrame(columns=["filename", "domain", "analysis_accession"])

        dl = {}
        if not source_meta.empty:
            dl = source_meta.set_index("filename")["domain"].to_dict()
        analysis_lookup = {}
        if not source_meta.empty:
            analysis_lookup = source_meta.set_index("filename")["analysis_accession"].to_dict()

        for fp in rumen_files:
            try:
                prev = read_fasta_preview(fp)
            except (EOFError, OSError) as e:
                print(f"SKIP rumen preview {fp}: {e}")
                continue
            sample = sanitize_sample_id(fp.stem.replace(".fa", ""))
            out_fp = proteome_dir / f"rumen__{sample}.faa"
            domain = dl.get(fp.name, "Bacteria")
            analysis_accession = analysis_lookup.get(fp.name, "")
            if prev["aa_like"] >= max(1, int(0.7 * prev["checked"])):
                try:
                    with gzip.open(fp, "rb") as s, out_fp.open("wb") as d:
                        shutil.copyfileobj(s, d)
                except (EOFError, OSError) as e:
                    print(f"SKIP rumen protein {fp}: {e}")
                    continue
                sample_records.append(
                    dict(
                        sample=out_fp.stem,
                        source="rumen",
                        ecosystem="rumen",
                        domain=domain,
                        source_analysis_accession=analysis_accession,
                        proteome_faa=str(out_fp),
                    )
                )
            elif CFG["allow_gene_calling_fallback"]:
                ensure_prodigal_available()
                try:
                    nuc = maybe_decompress_to_temp(fp, proteome_dir / "_tmp_rumen")
                except RuntimeError as e:
                    print(f"SKIP rumen nuc {fp}: {e}")
                    continue
                if not run_prodigal_safe(nuc, out_fp, "RUMEN"):
                    continue
                sample_records.append(
                    dict(
                        sample=out_fp.stem,
                        source="rumen",
                        ecosystem="rumen",
                        domain=domain,
                        source_analysis_accession=analysis_accession,
                        proteome_faa=str(out_fp),
                    )
                )

    sample_df = pd.DataFrame(sample_records)
    manifest_path = CFG["out_root"] / "proteome_sample_manifest.tsv"
    sample_df.to_csv(manifest_path, sep="\t", index=False)
    record_artifact(manifest_path, "metadata", "Prepared MAG-level proteome sample manifest")

    source_counts = (
        sample_df.groupby(["source", "ecosystem"], as_index=False).size().rename(columns={"size": "n_samples"})
        if not sample_df.empty
        else pd.DataFrame(columns=["source", "ecosystem", "n_samples"])
    )
    source_counts_path = CFG["out_root"] / "sample_source_counts.tsv"
    source_counts.to_csv(source_counts_path, sep="\t", index=False)
    record_artifact(source_counts_path, "metadata", "Sample counts by source/ecosystem for audit")

    print(f"Prepared {len(sample_df)} proteome samples")

In [ ]:
if OFFLINE_MODE:
    print("OFFLINE MODE — skipping ESM2 embedding.")
else:
    from Bio import SeqIO
    from methanet.embedding.esm2 import EmbeddingConfig, ESM2Embedder

    sample_df = pd.read_csv(CFG["out_root"] / "proteome_sample_manifest.tsv", sep="\t")
    if sample_df.empty:
        raise RuntimeError("No proteome samples; check data acquisition cells.")

    emb_cfg = EmbeddingConfig(
        model_name=CFG["esm2_model"],
        batch_size=CFG["esm2_batch_size"],
        max_length=CFG["esm2_max_length"],
        device=CFG["device"],
        fp16=False,
    )
    embedder = ESM2Embedder(emb_cfg)
    valid_aa = set("ACDEFGHIKLMNPQRSTVWYBXZJUO")

    def normalize_aa(seq: str) -> str:
        seq = seq.upper().replace("*", "").replace("-", "")
        return "".join(ch if ch in valid_aa else "X" for ch in seq)

    embeddings, kept_rows = [], []
    stats_emb = {
        "total": 0,
        "no_valid": 0,
        "empty_emb": 0,
        "non_finite": 0,
        "embedded": 0,
        "max_proteins_per_genome": int(CFG["max_proteins_per_genome"]),
    }

    for row in sample_df.itertuples(index=False):
        stats_emb["total"] += 1
        fp = Path(row.proteome_faa)
        seqs, ids = [], []
        with fp.open() as h:
            for rec in SeqIO.parse(h, "fasta"):
                seq = normalize_aa(str(rec.seq))
                if len(seq) < CFG["min_aa_len"]:
                    continue
                seqs.append(seq)
                ids.append(rec.id)
                if len(seqs) >= CFG["max_proteins_per_genome"]:
                    break
        if not seqs:
            stats_emb["no_valid"] += 1
            continue
        prot_emb = embedder.embed_proteins(seqs, ids)
        if not prot_emb:
            stats_emb["empty_emb"] += 1
            continue
        genome_emb = embedder.embed_genome(prot_emb, aggregation="mean").astype(np.float32)
        if not np.isfinite(genome_emb).all():
            stats_emb["non_finite"] += 1
            continue
        embeddings.append(genome_emb)
        kept_rows.append(dict(row._asdict(), n_proteins_used=len(seqs)))
        stats_emb["embedded"] += 1

    if not embeddings:
        raise RuntimeError(f"No valid embeddings produced. Stats: {stats_emb}")

    emb_mat = np.vstack(embeddings)
    meta_emb = pd.DataFrame(kept_rows)

    out_npz = CFG["out_root"] / "genome_embeddings.npz"
    np.savez_compressed(
        out_npz,
        embeddings=emb_mat,
        sample=meta_emb["sample"].values,
        source=meta_emb["source"].values,
        ecosystem=meta_emb["ecosystem"].values,
        domain=meta_emb["domain"].values,
        source_analysis_accession=meta_emb["source_analysis_accession"].astype(str).values,
        n_proteins_used=meta_emb["n_proteins_used"].values,
    )

    meta_path = CFG["out_root"] / "embedding_metadata.tsv"
    meta_emb.to_csv(meta_path, sep="\t", index=False)

    stats_path = CFG["out_root"] / "embedding_stats.json"
    with stats_path.open("w") as f:
        json.dump(stats_emb, f, indent=2)

    record_artifact(out_npz, "core", "ESM2 genome embedding matrix")
    record_artifact(meta_path, "core", "Per-genome embedding metadata")
    record_artifact(stats_path, "core", "Embedding run quality counters")

    print(f"Embedding matrix: {emb_mat.shape} | saved: {out_npz}")
    print("Stats:", stats_emb)

In [ ]:
npz_path = ARTIFACTS_DIR / "genome_embeddings.npz"
bundle = np.load(npz_path, allow_pickle=True)

X_raw = bundle["embeddings"].astype(np.float32)
meta = pd.DataFrame({
    "sample": bundle["sample"].astype(str),
    "source": bundle["source"].astype(str),
    "ecosystem": bundle["ecosystem"].astype(str),
    "domain": bundle["domain"].astype(str),
    "n_proteins_used": bundle["n_proteins_used"].astype(int),
})

if "source_analysis_accession" in bundle.files:
    meta["source_analysis_accession"] = bundle["source_analysis_accession"].astype(str)
else:
    meta["source_analysis_accession"] = ""

mask = np.isfinite(X_raw).all(axis=1)
X = X_raw[mask]
meta = meta.loc[mask].reset_index(drop=True)
n = len(meta)

print(f"Loaded {n} genome embeddings  shape={X.shape}")
print("Ecosystem:\n", meta["ecosystem"].value_counts().to_string())
print("Domain:\n", meta["domain"].value_counts().to_string())
print("Unique rumen analyses:", meta.loc[meta["source"] == "rumen", "source_analysis_accession"].nunique())

In [ ]:
if HAS_UMAP:
    print("Running UMAP …")
    umap_model = umap_pkg.UMAP(
        n_neighbors=min(CFG["umap_n_neighbors"], max(2, n - 1)),
        min_dist=CFG["umap_min_dist"],
        metric="cosine",
        random_state=SEED,
    )
    U = umap_model.fit_transform(X)
else:
    print("UMAP not available; using PCA(2) fallback for umap_* coordinates.")
    U = PCA(n_components=2, random_state=SEED).fit_transform(X)

print("Running t-SNE …")
tsne_perp = min(CFG["tsne_perplexity"], max(2, (n - 1) // 3))
T = TSNE(
    n_components=2,
    perplexity=tsne_perp,
    random_state=SEED,
    init="pca",
    learning_rate="auto",
).fit_transform(X)

print("Running HDBSCAN …")
hdb = HDBSCAN(min_cluster_size=max(5, n // 20), metric="euclidean")
cluster_labels = hdb.fit_predict(X)

meta["cluster"] = cluster_labels
meta["umap_1"] = U[:, 0]
meta["umap_2"] = U[:, 1]
meta["tsne_1"] = T[:, 0]
meta["tsne_2"] = T[:, 1]

# KNN bridging (entropy + mixing coefficient)
k = min(CFG["knn_k_bridge"], n)
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
nn_dists, nn_idx = nbrs.kneighbors(return_distance=True)

bridging_scores, mixing_coeffs = [], []
for i, nn in enumerate(nn_idx):
    eco = meta.loc[nn, "ecosystem"].values
    cnt = Counter(eco)
    probs = np.array([v / len(eco) for v in cnt.values()], dtype=float)
    entropy = max(0.0, float(-np.sum(probs * np.log2(probs + 1e-12))))
    bridging_scores.append(entropy)
    mixing_coeffs.append(float((eco != meta.loc[i, "ecosystem"]).mean()))

meta["bridging_score"] = bridging_scores
meta["mixing_coeff"] = mixing_coeffs

print(f"Cluster distribution:\n{meta['cluster'].value_counts().sort_index().to_string()}")
print(f"Bridge candidates (mixing>0): {(meta['mixing_coeff'] > 0).sum()}")

In [ ]:
non_noise = cluster_labels >= 0
metrics = {"n_samples": int(n), "status": "ok", "umap_available": bool(HAS_UMAP)}

if non_noise.sum() > 2 and len(set(cluster_labels[non_noise])) > 1:
    metrics["silhouette_non_noise"] = float(
        silhouette_score(X[non_noise], cluster_labels[non_noise], metric="euclidean")
    )
else:
    metrics["silhouette_non_noise"] = float("nan")


def purity(df, col):
    vals = []
    for _, sub in df[df["cluster"] >= 0].groupby("cluster"):
        c = sub[col].value_counts()
        vals.append(c.iloc[0] / c.sum())
    return float(np.mean(vals)) if vals else float("nan")


metrics["cluster_purity_ecosystem"] = purity(meta, "ecosystem")
metrics["cluster_purity_domain"] = purity(meta, "domain")
metrics["n_clusters_excluding_noise"] = int(len(set(cluster_labels[cluster_labels >= 0])))
metrics["noise_fraction"] = float((cluster_labels < 0).mean())
metrics["n_unique_rumen_analyses"] = int(
    meta.loc[meta["source"] == "rumen", "source_analysis_accession"].nunique()
)

bridge_df = meta.sort_values("bridging_score", ascending=False).head(min(100, len(meta))).copy()
proj_path = ARTIFACTS_DIR / "embedding_projection_clusters.tsv"
bridge_path = ARTIFACTS_DIR / "bridging_genomes_top.tsv"
metrics_path = ARTIFACTS_DIR / "poc_metrics.json"

meta.to_csv(proj_path, sep="\t", index=False)
bridge_df.to_csv(bridge_path, sep="\t", index=False)
with metrics_path.open("w") as f:
    json.dump(metrics, f, indent=2)

record_artifact(proj_path, "core", "UMAP/t-SNE/HDBSCAN projections with bridge scores")
record_artifact(bridge_path, "core", "Top bridge genomes ranked by entropy")
record_artifact(metrics_path, "core", "Core POC quality metrics")

print("Metrics:", json.dumps(metrics, indent=2))

In [ ]:
def _html_layout(fig):
    fig.update_layout(
        font=dict(size=13), plot_bgcolor="white", paper_bgcolor="white",
        xaxis=dict(showgrid=True, gridcolor="#eee"),
        yaxis=dict(showgrid=True, gridcolor="#eee"),
    )
    return fig

fig_umap_eco = _html_layout(px.scatter(
    meta, x="umap_1", y="umap_2", color="ecosystem", symbol="domain",
    hover_data=["sample", "cluster", "bridging_score", "n_proteins_used"],
    color_discrete_map=ECOSYSTEM_COLORS,
    title="UMAP — ESM2 Proteome Embeddings: Wetland MUCC vs Rumen PRJEB31266",
    width=1050, height=720,
))
fig_umap_eco.write_html(ARTIFACTS_DIR / "umap_ecosystem_domain.html")

fig_umap_cl = _html_layout(px.scatter(
    meta, x="umap_1", y="umap_2", color=meta["cluster"].astype(str), symbol="ecosystem",
    hover_data=["sample", "domain", "bridging_score"],
    title="UMAP — HDBSCAN Cluster Map",
    width=1050, height=720,
))
fig_umap_cl.write_html(ARTIFACTS_DIR / "umap_hdbscan_clusters.html")

fig_tsne = _html_layout(px.scatter(
    meta, x="tsne_1", y="tsne_2", color="ecosystem", symbol="domain",
    hover_data=["sample", "cluster", "bridging_score"],
    color_discrete_map=ECOSYSTEM_COLORS,
    title="t-SNE — ESM2 Proteome Embeddings: Wetland MUCC vs Rumen PRJEB31266",
    width=1050, height=720,
))
fig_tsne.write_html(ARTIFACTS_DIR / "tsne_ecosystem_domain.html")

print("Saved original HTML figures →", ARTIFACTS_DIR)


---
## Advanced Analytics

The cells below add statistical rigour and topological depth beyond the original POC.
All new artefacts are written to `ARTIFACTS_DIR` alongside the existing ones.


In [ ]:
# ── PCA: variance structure of the 1280-dim embedding space ──────────────────
n_comp = min(40, X.shape[1], n)
pca = PCA(n_components=n_comp, random_state=SEED)
X_pca = pca.fit_transform(X)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_pcs_80 = int(np.searchsorted(cumvar, 0.80)) + 1
n_pcs_90 = int(np.searchsorted(cumvar, 0.90)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree
ax = axes[0]
ax.bar(range(1, n_comp + 1), pca.explained_variance_ratio_ * 100,
       color="steelblue", alpha=0.8, edgecolor="white")
ax.axhline(1, color="red", ls="--", alpha=0.5, label="1% threshold")
ax.set_xlim(0.5, min(30, n_comp) + 0.5)
ax.set_xlabel("Principal Component", fontsize=12)
ax.set_ylabel("Variance Explained (%)", fontsize=12)
ax.set_title("PCA Scree — ESM2 Genome Embeddings (n=40, d=1280)", fontsize=12)
ax.legend()

# Cumulative
ax = axes[1]
ax.plot(range(1, n_comp + 1), cumvar * 100, "o-", color="steelblue", ms=5)
ax.axhline(80, color="orange", ls="--", alpha=0.7, label="80%")
ax.axhline(90, color="red",    ls="--", alpha=0.7, label="90%")
ax.set_xlim(0.5, min(30, n_comp) + 0.5); ax.set_ylim(0, 105)
ax.set_xlabel("Number of PCs", fontsize=12)
ax.set_ylabel("Cumulative Variance (%)", fontsize=12)
ax.set_title("Cumulative Variance", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "pca_variance_explained.png", dpi=160, bbox_inches="tight")
plt.close()

# PC1 × PC2
fig, ax = plt.subplots(figsize=(8, 6))
for eco, color in ECOSYSTEM_COLORS.items():
    m = meta["ecosystem"] == eco
    ax.scatter(X_pca[m, 0], X_pca[m, 1], c=color, label=eco.capitalize(),
               alpha=0.85, s=65, edgecolors="white", lw=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=12)
ax.set_title("PCA PC1 × PC2 — ESM2 Proteome Space", fontsize=12)
ax.legend(title="Ecosystem", fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "pca_pc1_pc2.png", dpi=160, bbox_inches="tight")
plt.close()

print(f"PCs for 80% variance: {n_pcs_80}")
print(f"PCs for 90% variance: {n_pcs_90}")
print(f"PC1 var: {pca.explained_variance_ratio_[0]*100:.1f}%  "
      f"PC1+PC2: {cumvar[1]*100:.1f}%")


In [ ]:
# ── Helper: compute 2-D KDE on a regular grid ─────────────────────────────────
def compute_kde_grid(pts, grid_size=180, pad=0.6):
    x, y = pts[:, 0], pts[:, 1]
    xi = np.linspace(x.min() - pad, x.max() + pad, grid_size)
    yi = np.linspace(y.min() - pad, y.max() + pad, grid_size)
    Xi, Yi = np.meshgrid(xi, yi)
    kde = gaussian_kde(pts.T, bw_method="scott")
    Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(grid_size, grid_size)
    return xi, yi, Zi


def static_kde_figure(c1, c2, meta_df, xlabel, ylabel, title, out_png):
    # Matplotlib KDE contour + scatter; annotates top-5 bridge candidates.
    fig, ax = plt.subplots(figsize=(11, 8))
    pts_all = np.column_stack([c1, c2])
    xi, yi, _ = compute_kde_grid(pts_all)
    for eco, cmap in [("wetland", "Blues"), ("rumen", "Reds")]:
        m = meta_df["ecosystem"] == eco
        pts = np.column_stack([c1[m], c2[m]])
        if len(pts) < 3: continue
        _, _, Zi = compute_kde_grid(pts)
        Zn = Zi / Zi.max()
        ax.contourf(xi, yi, Zn, levels=8, cmap=cmap, alpha=0.40)
        ax.contour (xi, yi, Zn, levels=5,
                    colors=ECOSYSTEM_COLORS[eco], alpha=0.55, linewidths=0.9)
    markers = {"wetland": "o", "rumen": "s"}
    for eco, color in ECOSYSTEM_COLORS.items():
        m = meta_df["ecosystem"] == eco
        ax.scatter(c1[m], c2[m], c=color, marker=markers[eco],
                   s=80, alpha=0.92, edgecolors="white", lw=0.7, zorder=6,
                   label=eco.capitalize())
    # Annotate top-5 bridge candidates
    top_idx = meta_df["mixing_coeff"].nlargest(5).index
    for i in top_idx:
        row = meta_df.loc[i]
        label = row["sample"].split("__")[-1][:12]
        ax.annotate(label, (c1[i], c2[i]),
                    xytext=(0, 14), textcoords="offset points", ha="center",
                    fontsize=7.5, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color="#333", lw=0.8),
                    bbox=dict(boxstyle="round,pad=0.25", fc="#ffe066", alpha=0.85))
    # Legend
    legend_elems = [
        mpatches.Patch(fc="#2E86AB", alpha=0.5, label="Wetland density"),
        mpatches.Patch(fc="#D62828", alpha=0.5, label="Rumen density"),
        plt.Line2D([0],[0], marker="o", color="w", mfc="#2E86AB", ms=10, label="Wetland genome"),
        plt.Line2D([0],[0], marker="s", color="w", mfc="#D62828", ms=10, label="Rumen genome"),
    ]
    ax.legend(handles=legend_elems, loc="upper right", fontsize=10, framealpha=0.9)
    ax.set_xlabel(xlabel, fontsize=13); ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.grid(True, alpha=0.18)
    plt.tight_layout()
    plt.savefig(out_png, dpi=180, bbox_inches="tight"); plt.close()
    print("Saved:", out_png)


# UMAP KDE (static PNG)
static_kde_figure(
    meta["umap_1"].values, meta["umap_2"].values, meta,
    "UMAP 1", "UMAP 2",
    "ESM2 Proteome Latent Space — UMAP + KDE Density Landscape\n"
    "(Wetland MUCC v2 × Rumen PRJEB31266 | annotated: top bridge candidates)",
    ARTIFACTS_DIR / "umap_kde_landscape.png",
)

# t-SNE KDE (static PNG)
static_kde_figure(
    meta["tsne_1"].values, meta["tsne_2"].values, meta,
    "t-SNE 1", "t-SNE 2",
    "ESM2 Proteome Latent Space — t-SNE + KDE Density Landscape\n"
    "(Wetland MUCC v2 × Rumen PRJEB31266 | annotated: top bridge candidates)",
    ARTIFACTS_DIR / "tsne_kde_landscape.png",
)


In [ ]:
def plotly_kde_figure(c1, c2, meta_df, dim1_lbl, dim2_lbl, title, out_html):
    fig = go.Figure()
    for eco, cmap in [("wetland", "Blues"), ("rumen", "Reds")]:
        m = meta_df["ecosystem"] == eco
        pts = np.column_stack([c1[m], c2[m]])
        if len(pts) < 3: continue
        xi, yi, Zi = compute_kde_grid(pts)
        Zn = Zi / Zi.max()
        fig.add_trace(go.Contour(
            x=xi, y=yi, z=Zn, colorscale=cmap,
            showscale=False, opacity=0.38,
            contours=dict(start=0.05, end=1.0, size=0.1),
            name=f"{eco.capitalize()} density", hoverinfo="skip",
        ))
    sym = {"wetland": "circle", "rumen": "square"}
    for eco, color in ECOSYSTEM_COLORS.items():
        m = meta_df["ecosystem"] == eco
        hover_txt = [
            (f"<b>{r['sample']}</b><br>Cluster: {r['cluster']}<br>"
             f"Bridge (entropy): {r['bridging_score']:.3f}<br>"
             f"Mixing coeff: {r['mixing_coeff']:.3f}<br>"
             f"Domain: {r['domain']}<br>Proteins: {r['n_proteins_used']}")
            for _, r in meta_df[m].iterrows()
        ]
        fig.add_trace(go.Scatter(
            x=c1[m], y=c2[m], mode="markers",
            marker=dict(color=color, symbol=sym[eco], size=11,
                        line=dict(color="white", width=1.2)),
            name=eco.capitalize(), text=hover_txt, hoverinfo="text",
        ))
    # Bridge annotations
    top_idx = meta_df["mixing_coeff"].nlargest(4).index
    for i in top_idx:
        row = meta_df.loc[i]
        fig.add_annotation(
            x=c1[i], y=c2[i],
            text=row["sample"].split("__")[-1][:14],
            showarrow=True, arrowhead=2, ax=0, ay=-40,
            font=dict(size=10, color="#222"),
            bgcolor="#ffe066", bordercolor="#888", opacity=0.88,
        )
    fig.update_layout(
        title=dict(text=title, font=dict(size=15)),
        xaxis_title=dim1_lbl, yaxis_title=dim2_lbl,
        width=1100, height=780,
        plot_bgcolor="white", paper_bgcolor="white",
        legend=dict(title="Layer", font=dict(size=12)),
    )
    fig.update_xaxes(showgrid=True, gridcolor="#eee")
    fig.update_yaxes(showgrid=True, gridcolor="#eee")
    fig.write_html(out_html)
    print("Saved:", out_html)


plotly_kde_figure(
    meta["umap_1"].values, meta["umap_2"].values, meta,
    "UMAP 1", "UMAP 2",
    "UMAP + KDE Landscape — ESM2 Proteome Space<br>"
    "<sup>Filled contours = ecosystem density; annotated = top bridge candidates</sup>",
    ARTIFACTS_DIR / "umap_kde_landscape.html",
)

plotly_kde_figure(
    meta["tsne_1"].values, meta["tsne_2"].values, meta,
    "t-SNE 1", "t-SNE 2",
    "t-SNE + KDE Landscape — ESM2 Proteome Space<br>"
    "<sup>Filled contours = ecosystem density; annotated = top bridge candidates</sup>",
    ARTIFACTS_DIR / "tsne_kde_landscape.html",
)


In [ ]:
# ── PERMANOVA — permutation test for ecosystem separation ─────────────────────
def permanova_test(X_mat, group_labels, n_perm=999, rng=None):
    """
    Anderson (2001) PERMANOVA / adonis.
    Returns: (F_statistic, p_value, R_squared)
    """
    if rng is None:
        rng = np.random.default_rng(42)
    D2 = squareform(pdist(X_mat, metric="euclidean")) ** 2
    unique_groups = np.unique(group_labels)
    g = len(unique_groups)
    n = len(group_labels)

    def pseudo_f(D2_, lbl):
        SS_T = D2_.sum() / (2 * n)
        SS_W = 0.0
        for grp in np.unique(lbl):
            idx = np.where(lbl == grp)[0]
            n_g = len(idx)
            if n_g < 2: continue
            SS_W += D2_[np.ix_(idx, idx)].sum() / (2 * n_g)
        SS_B = SS_T - SS_W
        F = (SS_B / (g - 1)) / (SS_W / (n - g))
        R2 = SS_B / SS_T
        return F, R2

    obs_F, obs_R2 = pseudo_f(D2, group_labels)
    perm_Fs = []
    for _ in range(n_perm):
        pF, _ = pseudo_f(D2, rng.permutation(group_labels))
        perm_Fs.append(pF)
    p_val = (np.sum(np.array(perm_Fs) >= obs_F) + 1) / (n_perm + 1)
    return float(obs_F), float(p_val), float(obs_R2)


eco_arr = meta["ecosystem"].values
print("Running PERMANOVA (999 permutations) …")
F_eco, p_eco, R2_eco = permanova_test(X, eco_arr, n_perm=999, rng=RNG)
print(f"PERMANOVA (ecosystem):  F={F_eco:.3f}  p={p_eco:.4f}  R²={R2_eco:.4f}")
print(f"  Interpretation: {'Ecosystem explains significant variance (p<0.05)' if p_eco < 0.05 else 'n.s.'}")

# Also test cluster assignment (non-noise only)
nn_mask = meta["cluster"] >= 0
if nn_mask.sum() > 4 and len(set(meta.loc[nn_mask, "cluster"])) > 1:
    F_cl, p_cl, R2_cl = permanova_test(
        X[nn_mask.values], meta.loc[nn_mask, "cluster"].values, n_perm=999, rng=RNG)
    print(f"PERMANOVA (HDBSCAN clusters): F={F_cl:.3f}  p={p_cl:.4f}  R²={R2_cl:.4f}")
else:
    F_cl = p_cl = R2_cl = float("nan")

# Visualise permutation distribution
perm_dist_data = []  # re-run a quick viz version
D2_viz = squareform(pdist(X, metric="euclidean")) ** 2
g_viz = len(np.unique(eco_arr)); n_viz = len(eco_arr)
perm_Fs_viz = []
for _ in range(500):
    lbl = RNG.permutation(eco_arr)
    SS_T = D2_viz.sum() / (2 * n_viz); SS_W = 0.0
    for grp in np.unique(lbl):
        idx = np.where(lbl == grp)[0]; n_g = len(idx)
        if n_g < 2: continue
        SS_W += D2_viz[np.ix_(idx, idx)].sum() / (2 * n_g)
    SS_B = SS_T - SS_W
    perm_Fs_viz.append((SS_B / (g_viz - 1)) / (SS_W / (n_viz - g_viz)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(perm_Fs_viz, bins=30, color="steelblue", alpha=0.75, edgecolor="white",
        label="Permuted F (n=500)")
ax.axvline(F_eco, color="#D62828", lw=2.5, label=f"Observed F = {F_eco:.1f}")
ax.set_xlabel("Pseudo-F statistic", fontsize=13)
ax.set_ylabel("Count", fontsize=13)
ax.set_title(
    f"PERMANOVA — Ecosystem Separation in ESM2 Space\n"
    f"F={F_eco:.2f}, p={p_eco:.4f}, R²={R2_eco:.3f}  (999 permutations)",
    fontsize=12, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "permanova_ecosystem.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved PERMANOVA figure")


In [ ]:
# ── Ecosystem trajectory: projection onto rumen → wetland axis ─────────────────
mean_w = X[meta["ecosystem"] == "wetland"].mean(axis=0)
mean_r = X[meta["ecosystem"] == "rumen"  ].mean(axis=0)
direction = mean_w - mean_r
direction_norm = direction / (np.linalg.norm(direction) + 1e-12)
proj_raw   = X @ direction_norm
proj_min, proj_max = proj_raw.min(), proj_raw.max()
proj_norm  = (proj_raw - proj_min) / (proj_max - proj_min + 1e-12)
meta["wetland_projection"] = proj_norm

t_stat, t_p = stats.ttest_ind(
    proj_norm[meta["ecosystem"] == "wetland"],
    proj_norm[meta["ecosystem"] == "rumen"],
)
print(f"t-test projection  t={t_stat:.3f}  p={t_p:.2e}")

# Static violin figure
fig, ax = plt.subplots(figsize=(10, 6))
for i, (eco, color) in enumerate(ECOSYSTEM_COLORS.items()):
    vals = proj_norm[meta["ecosystem"] == eco]
    parts = ax.violinplot(vals, positions=[i], widths=0.55,
                          showmeans=True, showextrema=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(color); pc.set_alpha(0.65)
    parts["cmeans"].set_color("black"); parts["cbars"].set_color(color)
    parts["cmaxes"].set_color(color);  parts["cmins"].set_color(color)
    jitter = RNG.uniform(-0.06, 0.06, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               c=color, s=45, alpha=0.85, edgecolors="white", zorder=5)
# Annotate top bridges
for j in meta["mixing_coeff"].nlargest(3).index:
    row = meta.loc[j]
    pos = 0 if row["ecosystem"] == "rumen" else 1
    ax.annotate(row["sample"].split("__")[-1][:12],
                (pos, row["wetland_projection"]),
                xytext=(pos + 0.17, row["wetland_projection"]),
                fontsize=8, va="center",
                arrowprops=dict(arrowstyle="->", color="#333", lw=0.8),
                bbox=dict(boxstyle="round,pad=0.2", fc="#ffe066", alpha=0.8))
ax.set_xticks([0, 1])
ax.set_xticklabels(["Rumen\n(PRJEB31266)", "Wetland\n(MUCC)"], fontsize=13)
ax.set_ylabel("Projection: rumen → wetland axis\n(0 = rumen pole · 1 = wetland pole)",
              fontsize=11)
ax.set_title(f"Ecosystem Trajectory in ESM2 Space  "
             f"(t={t_stat:.2f}, p={t_p:.1e})", fontsize=12, fontweight="bold")
ax.set_ylim(-0.07, 1.07); ax.grid(True, alpha=0.28, axis="y")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "ecosystem_trajectory.png", dpi=160, bbox_inches="tight")
plt.close()

# Interactive Plotly — UMAP coloured by trajectory score
fig_traj = px.scatter(
    meta, x="umap_1", y="umap_2",
    color="wetland_projection", color_continuous_scale="RdBu_r",
    symbol="ecosystem", symbol_map={"wetland": "circle", "rumen": "square"},
    hover_data=["sample", "cluster", "mixing_coeff", "domain",
                "n_proteins_used", "wetland_projection"],
    title=("UMAP — Projection onto Rumen → Wetland ESM2 Axis<br>"
           "<sup>Red = rumen pole · Blue = wetland pole · "
           f"t-test p={t_p:.1e}</sup>"),
    labels={"wetland_projection": "Wetland score"},
    width=1100, height=760,
)
fig_traj.update_layout(plot_bgcolor="white", paper_bgcolor="white", font=dict(size=13))
fig_traj.write_html(ARTIFACTS_DIR / "umap_trajectory_projection.html")
print("Saved trajectory figures")


In [ ]:
# ── Per-genome silhouette profiles ──────────────────────────────────────────────
nn_mask2 = meta["cluster"] >= 0
if nn_mask2.sum() > 2 and len(set(meta.loc[nn_mask2, "cluster"])) > 1:
    X_nn2    = X[nn_mask2.values]
    cl_nn2   = meta.loc[nn_mask2, "cluster"].values
    sil_vals = silhouette_samples(X_nn2, cl_nn2, metric="euclidean")

    sil_df = meta[nn_mask2].copy().reset_index(drop=True)
    sil_df["silhouette"] = sil_vals
    sil_df = sil_df.sort_values(["cluster", "silhouette"], ascending=[True, False])

    fig, ax = plt.subplots(figsize=(12, 6))
    colors_bar = [ECOSYSTEM_COLORS[e] for e in sil_df["ecosystem"]]
    ax.barh(range(len(sil_df)), sil_df["silhouette"],
            color=colors_bar, alpha=0.82, edgecolor="white", height=0.85)
    ax.axvline(metrics["silhouette_non_noise"], color="black", ls="--", lw=1.8,
               label=f"Mean = {metrics['silhouette_non_noise']:.3f}")
    # Cluster separators
    cumul = 0
    for cl, sub in sil_df.groupby("cluster", sort=True):
        cumul += len(sub)
        if cumul < len(sil_df):
            ax.axhline(cumul - 0.5, color="#888", ls=":", lw=0.9)
        ax.text(ax.get_xlim()[1] + 0.01, cumul - len(sub)/2 - 0.5,
                f"C{cl}", fontsize=8.5, fontweight="bold", va="center")
    ax.set_xlabel("Silhouette Score", fontsize=13)
    ax.set_ylabel("Genome (sorted by cluster)", fontsize=11)
    ax.set_title("Per-Genome Silhouette Scores — HDBSCAN Clusters\n"
                 "(blue = wetland | red = rumen)", fontsize=12, fontweight="bold")
    legend_h = [
        mpatches.Patch(color="#2E86AB", label="Wetland"),
        mpatches.Patch(color="#D62828", label="Rumen"),
        plt.Line2D([0],[0], ls="--", color="black",
                   label=f"Mean = {metrics['silhouette_non_noise']:.3f}"),
    ]
    ax.legend(handles=legend_h, fontsize=11)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "silhouette_profiles.png", dpi=160, bbox_inches="tight")
    plt.close()
    print("Saved silhouette profiles")
    print(sil_df[["sample","ecosystem","cluster","silhouette"]].to_string(index=False))
else:
    print("Insufficient non-noise samples for per-genome silhouette.")
    sil_df = pd.DataFrame()


In [ ]:
# ── Enhanced bridge-genome analysis ───────────────────────────────────────────
bridge_ranked = meta.sort_values("mixing_coeff", ascending=False).reset_index(drop=True)
bridge_ranked["rank"] = range(1, len(bridge_ranked) + 1)

# Interactive scatter: mixing_coeff × bridging entropy
fig_br = px.scatter(
    bridge_ranked,
    x="mixing_coeff", y="bridging_score",
    color="ecosystem", symbol="domain",
    hover_data=["sample", "cluster", "n_proteins_used", "rank", "wetland_projection"],
    color_discrete_map=ECOSYSTEM_COLORS,
    size="n_proteins_used", size_max=22,
    title=("Bridge Genome Candidates — Cross-Ecosystem Mixing × Neighbourhood Entropy<br>"
           "<sup>Top-right corner = most bridge-like genomes</sup>"),
    labels={
        "mixing_coeff":    "Cross-ecosystem mixing coefficient (fraction of KNN from other ecosystem)",
        "bridging_score":  "Neighbourhood entropy (bits, Shannon H)",
    },
    width=1100, height=680,
)
fig_br.update_layout(plot_bgcolor="white", paper_bgcolor="white", font=dict(size=13))
fig_br.update_xaxes(showgrid=True, gridcolor="#eee")
fig_br.update_yaxes(showgrid=True, gridcolor="#eee")
fig_br.write_html(ARTIFACTS_DIR / "bridge_genome_analysis.html")

top_cands = bridge_ranked[bridge_ranked["mixing_coeff"] > 0].head(10)
top_cands.to_csv(ARTIFACTS_DIR / "bridge_top_candidates.tsv", sep="\t", index=False)
print("\nTop bridge candidates (mixing_coeff > 0):")
cols = ["rank","sample","ecosystem","domain","cluster","mixing_coeff","bridging_score",
        "wetland_projection","n_proteins_used"]
print(top_cands[cols].to_string(index=False))


In [ ]:
# ── KNN neighbourhood composition of top bridge candidates ───────────────────
k_vis = min(10, n - 1)
nbrs_vis = NearestNeighbors(n_neighbors=k_vis).fit(X)
dists_vis, idx_vis = nbrs_vis.kneighbors(return_distance=True)

knn_records = []
for i in range(len(meta)):
    if meta.loc[i, "mixing_coeff"] < 0.05:
        continue
    for rank_nn, (ni, nd) in enumerate(zip(idx_vis[i], dists_vis[i])):
        knn_records.append({
            "query_genome":       meta.loc[i, "sample"],
            "query_ecosystem":    meta.loc[i, "ecosystem"],
            "query_mixing_coeff": meta.loc[i, "mixing_coeff"],
            "nn_rank":            rank_nn + 1,
            "nn_genome":          meta.loc[ni, "sample"],
            "nn_ecosystem":       meta.loc[ni, "ecosystem"],
            "nn_euclidean_dist":  float(nd),
            "cross_ecosystem":    meta.loc[i, "ecosystem"] != meta.loc[ni, "ecosystem"],
        })

knn_df = pd.DataFrame(knn_records)
knn_df.to_csv(ARTIFACTS_DIR / "bridge_knn_neighborhoods.tsv", sep="\t", index=False)
print(f"KNN neighbourhood table: {len(knn_df)} rows saved")

# Visualise neighbourhood composition as stacked bar
if not knn_df.empty:
    comp = (knn_df.groupby(["query_genome", "nn_ecosystem"])
            .size().unstack(fill_value=0).reset_index())
    comp.columns.name = None
    fig_knn = px.bar(
        comp.melt(id_vars="query_genome", var_name="nn_ecosystem", value_name="count"),
        x="query_genome", y="count", color="nn_ecosystem",
        color_discrete_map=ECOSYSTEM_COLORS,
        title="KNN Neighbourhood Composition of Bridge Candidates<br>"
              f"<sup>k={k_vis} nearest neighbours</sup>",
        labels={"query_genome": "Bridge candidate genome",
                "count": f"# of {k_vis}-NN"},
        width=1050, height=550,
    )
    fig_knn.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                          xaxis_tickangle=-40, font=dict(size=12))
    fig_knn.write_html(ARTIFACTS_DIR / "bridge_knn_composition.html")
    print("Saved KNN composition figure")


In [ ]:
# ── Pairwise cosine-distance heatmap ──────────────────────────────────────────
D_cos = cosine_distances(X)

# Sort: ecosystem, then cluster, then sample name
sort_order = meta.sort_values(["ecosystem", "cluster", "sample"]).index.values
D_sorted   = D_cos[np.ix_(sort_order, sort_order)]
eco_sorted  = meta.loc[sort_order, "ecosystem"].values
samp_sorted = meta.loc[sort_order, "sample"].values
n_w = (eco_sorted == "wetland").sum()

short_lbl = [
    (s.split("__")[-1][:14] if "__" in s else s[:14])
    for s in samp_sorted
]

fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(D_sorted, cmap="RdYlGn_r", aspect="auto",
               vmin=0, vmax=D_cos.max())
plt.colorbar(im, ax=ax, label="Cosine Distance", shrink=0.8, pad=0.01)

# Ecosystem boundary
ax.axhline(n_w - 0.5, color="black", lw=2.2, ls="--")
ax.axvline(n_w - 0.5, color="black", lw=2.2, ls="--")

ax.set_xticks(range(len(short_lbl)))
ax.set_xticklabels(short_lbl, rotation=90, fontsize=5.5)
ax.set_yticks(range(len(short_lbl)))
ax.set_yticklabels(short_lbl, fontsize=5.5)

ax.text(n_w / 2, -2.2, "WETLAND (MUCC)", ha="center",
        fontsize=10, fontweight="bold", color="#2E86AB")
ax.text(n_w + (len(sort_order) - n_w) / 2, -2.2, "RUMEN (PRJEB31266)", ha="center",
        fontsize=10, fontweight="bold", color="#D62828")

ax.set_title("Pairwise Cosine Distance — ESM2 Genome Embeddings (n=40 × 1280)\n"
             "Sorted by ecosystem; dashed line = ecosystem boundary",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "pairwise_cosine_heatmap.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved cosine distance heatmap")

# Summary stats
within_w  = D_cos[np.ix_(np.where(meta["ecosystem"]=="wetland")[0],
                          np.where(meta["ecosystem"]=="wetland")[0])]
within_r  = D_cos[np.ix_(np.where(meta["ecosystem"]=="rumen")[0],
                          np.where(meta["ecosystem"]=="rumen")[0])]
cross_wr  = D_cos[np.ix_(np.where(meta["ecosystem"]=="wetland")[0],
                          np.where(meta["ecosystem"]=="rumen")[0])]
print(f"Within-wetland   mean cosine dist: {within_w[within_w>0].mean():.4f}")
print(f"Within-rumen     mean cosine dist: {within_r[within_r>0].mean():.4f}")
print(f"Cross-ecosystem  mean cosine dist: {cross_wr.mean():.4f}")


In [ ]:
# ── 2×2 Proposal Panel Figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# ── A: UMAP + KDE ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
pts_all_u = np.column_stack([meta["umap_1"], meta["umap_2"]])
xi_u, yi_u, _ = compute_kde_grid(pts_all_u)
for eco, cmap in [("wetland", "Blues"), ("rumen", "Reds")]:
    m = meta["ecosystem"] == eco
    pts = np.column_stack([meta.loc[m,"umap_1"], meta.loc[m,"umap_2"]])
    if len(pts) < 3: continue
    _, _, Zi = compute_kde_grid(pts)
    ax.contourf(xi_u, yi_u, Zi/Zi.max(), levels=7, cmap=cmap, alpha=0.38)
markers_map = {"wetland": "o", "rumen": "s"}
for eco, color in ECOSYSTEM_COLORS.items():
    m = meta["ecosystem"] == eco
    ax.scatter(meta.loc[m,"umap_1"], meta.loc[m,"umap_2"],
               c=color, marker=markers_map[eco], s=75, alpha=0.92,
               edgecolors="white", lw=0.7, label=eco.capitalize(), zorder=5)
ax.set_title("A  UMAP + KDE Density Landscape", fontsize=12, fontweight="bold", loc="left")
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.legend(fontsize=10); ax.grid(True, alpha=0.18)

# ── B: PCA PC1×PC2 ────────────────────────────────────────────────────────────
ax = axes[0, 1]
for eco, color in ECOSYSTEM_COLORS.items():
    m = meta["ecosystem"] == eco
    ax.scatter(X_pca[m, 0], X_pca[m, 1], c=color, label=eco.capitalize(),
               alpha=0.88, s=70, edgecolors="white", lw=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=11)
ax.set_title("B  PCA: PC1 × PC2", fontsize=12, fontweight="bold", loc="left")
ax.legend(fontsize=10); ax.grid(True, alpha=0.28)

# ── C: t-SNE + KDE ────────────────────────────────────────────────────────────
ax = axes[1, 0]
pts_all_t = np.column_stack([meta["tsne_1"], meta["tsne_2"]])
xi_t, yi_t, _ = compute_kde_grid(pts_all_t)
for eco, cmap in [("wetland", "Blues"), ("rumen", "Reds")]:
    m = meta["ecosystem"] == eco
    pts = np.column_stack([meta.loc[m,"tsne_1"], meta.loc[m,"tsne_2"]])
    if len(pts) < 3: continue
    _, _, Zi = compute_kde_grid(pts)
    ax.contourf(xi_t, yi_t, Zi/Zi.max(), levels=7, cmap=cmap, alpha=0.38)
for eco, color in ECOSYSTEM_COLORS.items():
    m = meta["ecosystem"] == eco
    ax.scatter(meta.loc[m,"tsne_1"], meta.loc[m,"tsne_2"],
               c=color, marker=markers_map[eco], s=75, alpha=0.92,
               edgecolors="white", lw=0.7, label=eco.capitalize(), zorder=5)
ax.set_title("C  t-SNE + KDE Density Landscape", fontsize=12, fontweight="bold", loc="left")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=10); ax.grid(True, alpha=0.18)

# ── D: Ecosystem trajectory violin ────────────────────────────────────────────
ax = axes[1, 1]
for i, (eco, color) in enumerate(ECOSYSTEM_COLORS.items()):
    vals = proj_norm[meta["ecosystem"] == eco]
    parts = ax.violinplot(vals, positions=[i], widths=0.55,
                          showmeans=True, showextrema=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(color); pc.set_alpha(0.65)
    parts["cmeans"].set_color("black"); parts["cbars"].set_color(color)
    parts["cmaxes"].set_color(color);  parts["cmins"].set_color(color)
    jitter = RNG.uniform(-0.06, 0.06, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               c=color, s=42, alpha=0.88, edgecolors="white", zorder=5)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Rumen", "Wetland"], fontsize=12)
ax.set_ylabel("Rumen → Wetland Projection Score", fontsize=11)
ax.set_title("D  Ecosystem Trajectory", fontsize=12, fontweight="bold", loc="left")
ax.set_ylim(-0.07, 1.07); ax.grid(True, alpha=0.28, axis="y")

fig.suptitle(
    "MethaNet Blue Catalyst POC — ESM2 Proteome Embedding Analytics\n"
    f"40 genomes (20 MUCC wetland + 20 rumen PRJEB31266) · ESM2-650M (d=1280)\n"
    f"PERMANOVA: F={F_eco:.1f}, p={p_eco:.4f}, R²={R2_eco:.3f}  ·  "
    f"Silhouette (non-noise)={metrics['silhouette_non_noise']:.3f}  ·  "
    f"Cluster purity={metrics['cluster_purity_ecosystem']:.2f}",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "proposal_panel_figure.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved proposal panel figure →", ARTIFACTS_DIR / "proposal_panel_figure.png")


In [ ]:
# ── Compile advanced analytics summary + artifact manifests ───────────────────
within_w2 = D_cos[
    np.ix_(
        np.where(meta["ecosystem"] == "wetland")[0],
        np.where(meta["ecosystem"] == "wetland")[0],
    )
]
within_r2 = D_cos[
    np.ix_(
        np.where(meta["ecosystem"] == "rumen")[0],
        np.where(meta["ecosystem"] == "rumen")[0],
    )
]
cross_wr2 = D_cos[
    np.ix_(
        np.where(meta["ecosystem"] == "wetland")[0],
        np.where(meta["ecosystem"] == "rumen")[0],
    )
]

top_bridge_row = meta.sort_values("mixing_coeff", ascending=False).iloc[0]

advanced_summary = {
    "run_metadata": {
        "run_id": CFG["run_id"],
        "artifacts_dir": str(ARTIFACTS_DIR),
        "offline_mode": OFFLINE_MODE,
        "n_genomes": int(n),
        "embedding_dim": int(X.shape[1]),
        "n_wetland": int((meta["ecosystem"] == "wetland").sum()),
        "n_rumen": int((meta["ecosystem"] == "rumen").sum()),
        "n_unique_rumen_analyses": int(
            meta.loc[meta["source"] == "rumen", "source_analysis_accession"].nunique()
        ),
    },
    "poc_metrics": metrics,
    "pca": {
        "n_pcs_80pct_variance": n_pcs_80,
        "n_pcs_90pct_variance": n_pcs_90,
        "pc1_variance_pct": float(pca.explained_variance_ratio_[0] * 100),
        "pc2_variance_pct": float(pca.explained_variance_ratio_[1] * 100),
        "pc1_pc2_cumvar_pct": float(cumvar[1] * 100),
    },
    "permanova_ecosystem": {
        "F_statistic": F_eco,
        "p_value": p_eco,
        "R_squared": R2_eco,
        "n_permutations": 999,
        "significant_p05": p_eco < 0.05,
    },
    "permanova_clusters": {
        "F_statistic": F_cl,
        "p_value": p_cl,
        "R_squared": R2_cl,
    },
    "cosine_distances": {
        "within_wetland_mean": float(within_w2[within_w2 > 0].mean()),
        "within_rumen_mean": float(within_r2[within_r2 > 0].mean()),
        "cross_ecosystem_mean": float(cross_wr2.mean()),
        "separation_ratio": float(
            cross_wr2.mean() / max(within_w2[within_w2 > 0].mean(), within_r2[within_r2 > 0].mean(), 1e-9)
        ),
    },
    "ecosystem_trajectory": {
        "t_statistic": float(t_stat),
        "p_value": float(t_p),
        "wetland_mean_proj": float(proj_norm[meta["ecosystem"] == "wetland"].mean()),
        "rumen_mean_proj": float(proj_norm[meta["ecosystem"] == "rumen"].mean()),
        "n_rumen_in_wetland_pole_gt05": int(
            ((meta["ecosystem"] == "rumen") & (meta["wetland_projection"] > 0.5)).sum()
        ),
    },
    "bridging": {
        "n_candidates_mixing_gt0": int((meta["mixing_coeff"] > 0).sum()),
        "n_candidates_mixing_gt010": int((meta["mixing_coeff"] > 0.10).sum()),
        "top_bridge_genome": top_bridge_row["sample"],
        "top_bridge_mixing": float(top_bridge_row["mixing_coeff"]),
        "top_bridge_entropy": float(top_bridge_row["bridging_score"]),
    },
}

out_summary = ARTIFACTS_DIR / "advanced_analytics_summary.json"
with out_summary.open("w") as f:
    json.dump(advanced_summary, f, indent=2)
record_artifact(out_summary, "summary", "Machine-readable advanced analytics summary")

# Artifact manifest: exhaustive listing for downstream packaging/provenance.
description_map = {
    "genome_embeddings.npz": "ESM2 genome embedding matrix",
    "embedding_metadata.tsv": "Per-genome embedding metadata",
    "embedding_projection_clusters.tsv": "Projection, clustering, and bridge metrics",
    "bridging_genomes_top.tsv": "Top bridge genomes by entropy",
    "bridge_top_candidates.tsv": "Top bridge genomes by mixing coefficient",
    "poc_metrics.json": "Core POC metrics",
    "advanced_analytics_summary.json": "Advanced analytics machine-readable summary",
    "umap_ecosystem_domain.html": "Interactive UMAP by ecosystem/domain",
    "umap_hdbscan_clusters.html": "Interactive UMAP by HDBSCAN cluster",
    "tsne_ecosystem_domain.html": "Interactive t-SNE by ecosystem/domain",
    "umap_kde_landscape.png": "UMAP KDE static landscape",
    "umap_kde_landscape.html": "UMAP KDE interactive landscape",
    "tsne_kde_landscape.png": "t-SNE KDE static landscape",
    "tsne_kde_landscape.html": "t-SNE KDE interactive landscape",
    "pca_variance_explained.png": "PCA scree and cumulative variance",
    "pca_pc1_pc2.png": "PCA PC1 vs PC2 scatter",
    "permanova_ecosystem.png": "PERMANOVA permutation distribution",
    "ecosystem_trajectory.png": "Ecosystem trajectory violin plot",
    "umap_trajectory_projection.html": "Interactive UMAP with trajectory projection",
    "silhouette_profiles.png": "Per-genome silhouette profiles",
    "bridge_genome_analysis.html": "Bridge-genome scatter",
    "bridge_knn_neighborhoods.tsv": "KNN neighborhood table for bridge candidates",
    "bridge_knn_composition.html": "Bridge KNN composition interactive bar chart",
    "pairwise_cosine_heatmap.png": "Pairwise cosine distance heatmap",
    "proposal_panel_figure.png": "Composite 2x2 proposal panel",
    "prjeb31266_analysis_manifest.tsv": "Raw ENA analysis manifest",
    "prjeb31266_selected_subset.tsv": "Selected diversity-aware rumen subset",
    "proteome_sample_manifest.tsv": "Prepared proteome manifest",
    "sample_source_counts.tsv": "Counts by source and ecosystem",
}

artifact_rows = []
for p in sorted(ARTIFACTS_DIR.glob("*")):
    if not p.is_file():
        continue
    suffix = p.suffix.lower()
    if suffix in {".png", ".html"}:
        category = "figure"
    elif suffix in {".tsv", ".csv"}:
        category = "table"
    elif suffix in {".json", ".npz"}:
        category = "data"
    else:
        category = "other"

    artifact_rows.append(
        {
            "file": p.name,
            "relative_path": str(p.relative_to(ARTIFACTS_DIR)),
            "category": category,
            "description": description_map.get(p.name, "Generated by notebook"),
            "size_bytes": p.stat().st_size,
        }
    )

manifest_df = pd.DataFrame(artifact_rows)
manifest_tsv = ARTIFACTS_DIR / "artifact_manifest.tsv"
manifest_json = ARTIFACTS_DIR / "artifact_manifest.json"
manifest_df.to_csv(manifest_tsv, sep="\t", index=False)
manifest_df.to_json(manifest_json, orient="records", indent=2)
record_artifact(manifest_tsv, "summary", "Tabular artifact manifest")
record_artifact(manifest_json, "summary", "JSON artifact manifest")

registry_df = pd.DataFrame(ARTIFACT_RECORDS).drop_duplicates(subset=["relative_path"]) if ARTIFACT_RECORDS else pd.DataFrame(columns=["file", "relative_path", "category", "description"])
registry_path = ARTIFACTS_DIR / "artifact_registry.tsv"
registry_df.to_csv(registry_path, sep="\t", index=False)

print("Advanced analytics summary written →", out_summary)
print("Artifact manifests written →", manifest_tsv, "and", manifest_json)
print("Notebook artifact registry →", registry_path)
print(json.dumps(advanced_summary, indent=2))

---
## Artefacts produced by this notebook

### Core outputs
| File | Description |
|---|---|
| `genome_embeddings.npz` | ESM2 embedding matrix (MAG-level proteomes) |
| `embedding_metadata.tsv` | Per-genome metadata and proteins used |
| `embedding_projection_clusters.tsv` | UMAP / t-SNE / HDBSCAN and bridge metrics |
| `bridging_genomes_top.tsv` | Top bridge candidates (entropy-ranked) |
| `poc_metrics.json` | Core quality metrics |

### Expanded analytics
| File | Description |
|---|---|
| `pca_variance_explained.png` | Scree + cumulative variance |
| `pca_pc1_pc2.png` | PCA scatter PC1 × PC2 |
| `umap_kde_landscape.png` | UMAP with KDE contours (static) |
| `umap_kde_landscape.html` | UMAP with KDE contours (interactive) |
| `tsne_kde_landscape.png` | t-SNE with KDE contours (static) |
| `tsne_kde_landscape.html` | t-SNE with KDE contours (interactive) |
| `permanova_ecosystem.png` | PERMANOVA permutation distribution |
| `ecosystem_trajectory.png` | Violin: rumen→wetland projection |
| `umap_trajectory_projection.html` | Interactive UMAP coloured by trajectory |
| `silhouette_profiles.png` | Per-genome silhouette bar chart |
| `bridge_genome_analysis.html` | Bridge scatter: mixing × entropy |
| `bridge_top_candidates.tsv` | Top bridge candidates table |
| `bridge_knn_neighborhoods.tsv` | KNN composition of bridge candidates |
| `bridge_knn_composition.html` | Stacked bar: bridge KNN composition |
| `pairwise_cosine_heatmap.png` | Pairwise cosine distance heatmap |
| `proposal_panel_figure.png` | 2×2 summary panel for proposal |
| `advanced_analytics_summary.json` | Consolidated machine-readable stats summary |

### Provenance + packaging support
| File | Description |
|---|---|
| `prjeb31266_analysis_manifest.tsv` | Full ENA analysis manifest |
| `prjeb31266_selected_subset.tsv` | Diversity-aware rumen selection |
| `proteome_sample_manifest.tsv` | MAG proteome file manifest |
| `sample_source_counts.tsv` | Sample counts by source/ecosystem |
| `artifact_manifest.tsv` | Exhaustive artifact list (tabular) |
| `artifact_manifest.json` | Exhaustive artifact list (JSON) |
| `artifact_registry.tsv` | Notebook-registered artifact log |

This artifact set is designed to support reproducibility audits and downstream packaging.